# create positions from tissue boundary

In [1]:
# ── Imports ──────────────────────────────────────────────────────────
import sys, numpy as np
from shapely.geometry import Polygon
import matplotlib.pyplot as plt

sys.path.insert(0, "/path/to/merfish_pipeline/src")

from merfish_pipeline.common.config        import ExperimentConfig
from merfish_pipeline.common.io            import save_positions_array
from merfish_pipeline.acquisition.positions import (
    create_grid_positions, generate_scanning_path,
    load_hole_polygons, filter_scanning_path,
    close_scanning_path, get_path_stats,
)
from merfish_pipeline.visualization        import plot_fov_layout

ModuleNotFoundError: No module named 'shapely'

In [ ]:
# ── Config ───────────────────────────────────────────────────────────
ROOT = "/data/experiments/LT026_sample_03"
INPUT  = f"{ROOT}/metadata/input"
OUTPUT = f"{ROOT}/metadata"

config = ExperimentConfig(
    data_dir       = f"{ROOT}/images",
    metadata_dir   = f"{ROOT}/metadata",
    analysis_dir   = f"{ROOT}/analysis",
    round_info_csv = f"{ROOT}/metadata/round_info.csv",
    positions_txt  = f"{ROOT}/metadata/positions.txt",
    pixel_size_um          = 0.109,
    image_size_px          = 2048,
    non_overlap_fraction   = 0.9,
)
print(f"Step size: {config.step_size_um:.1f} µm")

In [ ]:
# ── Load boundary ────────────────────────────────────────────────────
import csv
with open(f"{INPUT}/boundary_positions.txt") as fh:
    boundary_pts = [(float(r[0]), float(r[1])) for r in csv.reader(fh)]

boundary = Polygon(boundary_pts)
xmin, ymin, xmax, ymax = boundary.bounds
cx, cy = boundary.centroid.x, boundary.centroid.y

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(*boundary.exterior.xy, ".-")
ax.plot(cx, cy, "o"); ax.invert_yaxis(); ax.axis("equal"); plt.show()

In [ ]:
# ── Build and filter grid ────────────────────────────────────────────
grid, xs, ys = create_grid_positions(
    cx, cy, xmin, ymin, xmax, ymax, config.step_size_um
)
path       = generate_scanning_path(grid, direction="vertical")
holes      = load_hole_polygons(INPUT)
filtered   = filter_scanning_path(
    path, boundary, holes, dilate=config.step_size_um / 2
)
reordered, _ = close_scanning_path(filtered, config.step_size_um, return_side="top")

total_um, max_step_um = get_path_stats(reordered)
print(f"{reordered.shape[0]} FOVs  |  path {total_um/1000:.1f} mm  |"
      f"  max step {max_step_um:.0f} µm")

In [ ]:
# ── Save positions ───────────────────────────────────────────────────
save_positions_array(reordered, f"{OUTPUT}/positions.txt")
print(f"Written: {OUTPUT}/positions.txt")